In [1]:
# conda env 'at3d_k9'
"""This notebook converts MISR observations (output from MISR_preprocess1.ipynb) 
to radiative transfer grid coordinates and applies wind correction.

Generates RTE*.nc and AT3D_OBS*.nc files.
"""

from pathlib import Path
PROJECT_ROOT = Path().resolve().parent.parent
INTERIM_DATA = PROJECT_ROOT / 'data' / 'interim'

import sys
import os
sys.path.append(str(PROJECT_ROOT))

import xarray as xr
import numpy as np
import at3d
import src.preprocess.MISR_preprocess2 as pp
import src.utils.util as util

In [2]:
# INPUT
# MISR NetCDF File (from MISR_preprocess1)
misr_data_file = INTERIM_DATA / 'MISR_OBS_I_CAMP2Ex_RF17_default.nc'

# Observation wavelength
wavelength = 0.67

# Cloud motion wind vector
s = 0.67 # m/s
theta = 34.8 # degrees
u = s * np.sin(np.radians(theta)) # m/s
v = s * np.cos(np.radians(theta)) # m/s

# OUTPUT
rte_grid_outfile = INTERIM_DATA / 'RTG_default.nc'
obs_outfile = INTERIM_DATA / 'AT3D_OBS_I_CAMP2Ex_RF17_default.nc'

In [3]:
# Create radiative transfer grid (for the bounding box)
if(os.path.exists(rte_grid_outfile)):
    print('Grid exists already. Loaded from {}'.format(rte_grid_outfile))
    rte_grid = xr.load_dataset(rte_grid_outfile)
else:
    z = np.array([0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4])
    rte_grid = at3d.grid.make_grid(0.275, 65, 0.275, 65, z)
    rte_grid.to_netcdf(rte_grid_outfile)
    print('Saved RTE grid to: {}'.format(rte_grid_outfile))

# Read in MISR data in Local North coordinates
misr_data_LN = xr.load_dataset(misr_data_file)

# Convert MISR data to ENU coordinates
misr_data_ENU = pp.LN2ENU(misr_data_LN)

# Horizontal shift to place origin at the corner of the rte grid (maintain centering)
misr_data_ENU['E'] = misr_data_ENU.E + np.max(rte_grid.x.data)/2
misr_data_ENU['E'] = misr_data_ENU.E.assign_attrs(units='km')
misr_data_ENU['N'] = misr_data_ENU.N + np.max(rte_grid.y.data)/2
misr_data_ENU['N'] = misr_data_ENU.N.assign_attrs(units='km')

# Project rays to the top of the domain
U2 = np.ones(misr_data_ENU.mu.data.shape)*np.max(rte_grid.z.data)
misr_data_ENU = pp.project_to_U2(misr_data_ENU, U2)

# Perform wind correction
misr_data_ENU = misr_data_ENU.assign_attrs(u=u, v=v)
misr_data_ENU['dt'] = (['camera'], -1*(np.array([0, 60, 113, 159, 204, 249, 295, 348, 408])-204))
misr_data_ENU['dt'] = misr_data_ENU.dt.assign_attrs(units='s')
misr_data_ENU['dt'] = misr_data_ENU.dt.assign_attrs(description='Time shift needed to register at AN camera')
misr_data_ENU['dE'] = misr_data_ENU.dt * u * 1e-3
misr_data_ENU['dE'] = misr_data_ENU.dE.assign_attrs(units='km')
misr_data_ENU['dN'] = misr_data_ENU.dt * v * 1e-3
misr_data_ENU['dN'] = misr_data_ENU.dN.assign_attrs(units='km')
misr_data_ENU['E2'] = misr_data_ENU.E2 + misr_data_ENU.dE
misr_data_ENU['N2'] = misr_data_ENU.N2 + misr_data_ENU.dN

# Calculate the maximum extent at oblique angles and pad
pad = 2 # km
min_extent, max_extent = pp.TOA_extent(rte_grid, misr_data_ENU)
min_extent = min_extent - pad
max_extent = max_extent + pad

# Store observations in a sensor dict
measurements = at3d.containers.SensorsDict()
for cam in misr_data_ENU.camera.data:
    measurements.add_sensor('MISR', pp.make_sensor(misr_data_ENU, cam, min_extent, max_extent, wavelength))

# Save measurements
util.save_sensor_dict(obs_outfile, measurements)

Grid exists already. Loaded from /data/gdi/b/johntl3/LundstromPub2025/data/interim/RTG_default.nc
Saved sensors to: /data/gdi/b/johntl3/LundstromPub2025/data/interim/AT3D_OBS_I_CAMP2Ex_RF17_default.nc
